# LAB | Abstractive Question Answering

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [ ]:
#!pip install -U langchain langchain-core langchain-classic langchain-pinecone langchain-huggingface datasets pinecone-client sentence-transformers torch

In [ ]:
#!pip uninstall -y pyarrow datasets numpy pandas transformers sentence-transformers huggingface_hub -q
#!pip install numpy==1.26.4 "pyarrow==14.0.2" "datasets==2.18.0" transformers sentence-transformers huggingface_hub -U -q

# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [3]:
# load the dataset from huggingface in streaming mode and shuffle it
#from datasets import load_dataset
#wiki_data = load_dataset(
#    'vblagoje/wikipedia_snippets_streamed',
#    split='train',
#    streaming=True
#).shuffle(seed=960)

from datasets import load_dataset
wiki_data = load_dataset(
    'wikimedia/wikipedia',
    '20231101.en',
    split='train',
    streaming=True
).shuffle(seed=960)

README.md: 0.00B [00:00, ?B/s]

c:\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\g\.cache\huggingface\hub\datasets--wikimedia--wikipedia. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [4]:
# show the contents of a single document in the dataset
next(iter(wiki_data))

{'id': '2097858',
 'url': 'https://en.wikipedia.org/wiki/Shawn%20Desman',
 'title': 'Shawn Desman',
 'text': 'Shawn Bosco Fernandes (born January 12, 1982), known by his stage name Shawn Desman, is a Canadian singer, songwriter, dancer, and choreographer.\n\nEarly life\nShawn Desman attended St. Francis of Assisi Elementary School in Toronto. After discovering his passion for performing, Desman\'s parents encouraged him to pursue a career in music. From age nine to 16, he made four Portuguese albums under his legal name, Shawn Fernandes. In 1998, Shawn was in The Boomtang Boys video for their song "Squeeze Toy" as a dancer. According to his own account, he adopted the name Desman because in his youth his friends referred to him as "Dez, man" (based on the ending of his last name - Fernandes) .\n\nCareer\n\n2002–2005: Debut Album and Back for More \nAt 18, Desman was signed to BMG Music Canada and started recording an album, which was released in 2002. The self-titled breakthrough album

In [5]:
# The 'wiki_snippets' dataset does not have 'section_title', so we will proceed without this specific filter
history = wiki_data

Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [6]:
from tqdm.auto import tqdm  # progress bar

total_doc_count = 50000 # you can consider 10000 also 

counter = 0
docs = []
# iterate through the dataset and apply our filter
for d in tqdm(history, total=total_doc_count):
    # extract the fields we need - title and passage text
    docs.append({
        'article_title': d['title'],
        'passage_text': d['text'][:500]  # trim to first 500 chars as a passage
    })
    # increase the counter on every iteration
    counter += 1
    if counter == total_doc_count:
        break

  0%|          | 0/50000 [00:00<?, ?it/s]

In [7]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
df.head()

,article_title,passage_text
0,Shawn Desman,"Shawn Bosco Fernandes (born January 12, 1982),..."
1,"Edward Seymour, 8th Duke of Somerset","Edward Seymour, 8th Duke of Somerset (December..."
2,PG-13 (professional wrestling),PG-13 was an Australian-American tag team comp...
3,The Headbangers,The Headbangers are a professional wrestling t...
4,Ivan Asen III of Bulgaria,"Ivan Asen III (, also Йоан Асен III, Ioan Asen..."


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [ ]:
#!pip uninstall -y pinecone-client pinecone -q
#!pip install "pinecone[grpc]" -U -q  # Latest v5 with gRPC

In [8]:
#from google.colab import userdata
import os
from pinecone import Pinecone
from pinecone import ServerlessSpec

# initialize connection to pinecone (get API key at app.pinecone.io)
pinecone_api_key = os.environ.get('PINECONE_API_KEY') or 'PINECONE_API_KEY'


Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [9]:
spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# connect to pinecone environment
pc = Pinecone(api_key=pinecone_api_key, environment=spec.region)

Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [10]:

index_name = "abstractive-question-answering" #give your index a meaningful name

import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=spec
    )
    time.sleep(1)

# connect to index
index = pc.Index(index_name)
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '150',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 27 Feb 2026 11:24:17 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '41',
                                    'x-pinecone-request-latency-ms': '40',
                                    'x-pinecone-response-duration-ms': '42'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [11]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# load the retriever model from huggingface model hub
retriever = SentenceTransformer('flax-sentence-embeddings/all_datasets_v3_mpnet-base', device=device) #load the retriever model from HuggingFace. Use the flax-sentence-embeddings/all_datasets_v3_mpnet-base model
retriever

W0227 12:24:58.804000 5848 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Could not find the bitsandbytes CUDA binary at WindowsPath('c:/Python/Python314/Lib/site-packages/bitsandbytes/libbitsandbytes_cuda130.dll')
The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\g\.cache\huggingface\hub\models--flax-sentence-embeddings--all_datasets_v3_mpnet-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [12]:
#from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

#You will create embedding for the passage_text variable and be use to include the meta data in each batch
for i in tqdm(range(0, len(df), batch_size)):
    # find end of batch
    i_end = min(i + batch_size, len(df))
    # extract batch
    batch = df.iloc[i:i_end]
    # generate embeddings for batch
    emb = retriever.encode(batch['passage_text'].tolist()).tolist()
    # upsert to pinecone
    index.upsert(vectors=zip(
        [str(x) for x in range(i, i_end)],
        emb,
        batch.to_dict(orient='records')
    ))

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/782 [00:00<?, ?it/s]

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '189',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 27 Feb 2026 11:40:17 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '37',
                                    'x-pinecone-request-latency-ms': '36',
                                    'x-pinecone-response-duration-ms': '39'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 50000}},
 'storageFullness': 0.0,
 'total_vector_count': 50000,
 'vector_type': 'dense'}

# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [13]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

c:\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\g\.cache\huggingface\hub\models--vblagoje--bart_lfqa. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [14]:
def query_pinecone(query, top_k):
    # generate embeddings for the query
    xq = retriever.encode(query).tolist()
    # search pinecone index for context passage with the answer
    xc = index.query(vector=xq, top_k=top_k, include_metadata=True)
    return xc

In [25]:
def format_query(query, context):
    # extract passage_text from Pinecone search result and add the <P> tag
    context = [f"<P> {m['metadata']['passage_text']}" for m in context]
    # concatenate all context passages
    context = " ".join(context)
    # concatenate the query and context passages
    query = f"question: {query} context: {context}"
    return query

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [16]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

QueryResponse(matches=[{'id': '24215',
 'metadata': {'article_title': 'FirstEnergy',
              'passage_text': 'FirstEnergy Corp is an electric utility '
                              'headquartered in Akron, Ohio. It was '
                              'established when Ohio Edison merged with '
                              'Centerior Energy in 1997. Its subsidiaries and '
                              'affiliates are involved in the distribution, '
                              'transmission, and generation of electricity, as '
                              'well as energy management and other '
                              'energy-related services. Its ten electric '
                              'utility operating companies comprise one of the '
                              "United States' largest investor-owned "
                              'utilities, based on serving 6 million customers '
                              'within a  area of Ohio, Pennsy'},
 'score': 0.48963

In [17]:
from pprint import pprint

In [19]:
# format the query in the form generator expects the input
query = format_query(query, result)
pprint(query)

('question: when was the first electric power system built? context: <P> '
 'FirstEnergy Corp is an electric utility headquartered in Akron, Ohio. It was '
 'established when Ohio Edison merged with Centerior Energy in 1997. Its '
 'subsidiaries and affiliates are involved in the distribution, transmission, '
 'and generation of electricity, as well as energy management and other '
 'energy-related services. Its ten electric utility operating companies '
 "comprise one of the United States' largest investor-owned utilities, based "
 'on serving 6 million customers within a  area of Ohio, Pennsy')


The output looks great. Now let's write a function to generate answers.

In [20]:
def generate_answer(query):
    # tokenize the query to get input_ids
    inputs = tokenizer([query], max_length=1024, return_tensors="pt").to(device)
    # use generator to predict output ids
    ids = generator.generate(inputs["input_ids"], num_beams=2, min_length=20, max_length=40)
    # use tokenizer to decode the output ids
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return pprint(answer)

In [21]:
generate_answer(query)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


("I'm not sure if this is what you're looking for, but I can tell you that the "
 'first electric power system was built in the early 19th century in the '
 'United States. The')


As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [26]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The first wireless message was sent in 1901 by Guglielmo Marconi, using a '
 'signal lamp to send a message from Poldhu, Cornwall, United Kingdom to '
 'London. The')


To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [27]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

The  automatic curb sender was a kind of telegraph key, invented by William Thomson, 1st Baron Kelvin for sending messages on a submarine communications cable, as the well-known Wheatstone transmitter sends them on a land line.

In both instruments, the signals are sent by means of a perforated ribbon of paper but the cable sender was the more complicated, because the cable signals are formed by both positive and negative currents, and not merely by a single current, whether positive or negative
---
Cabot Tower is a tower in St. John's, Newfoundland and Labrador, situated on Signal Hill. Construction of the tower began in 1898 to commemorate the 400th anniversary of John Cabot's landing in Newfoundland, and Queen Victoria's Diamond Jubilee.

In 1901, Guglielmo Marconi received the first trans-Atlantic wireless message at a position near the tower, the letter "S" in Morse Code sent from Poldhu, Cornwall, United Kingdom.  Cabot Tower is now the centre of the Signal Hill National Historic

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [28]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('COVID-19 is a new strain of the HIV virus. It is a new virus that has not '
 'yet been isolated from the human population. It is believed to have '
 'originated in the Congo')


In [29]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

AIDS is caused by a human immunodeficiency virus (HIV), which originated in non-human primates
in Central and West Africa. While various sub-groups of the virus acquired human infectivity at different times, the present pandemic had its origins in the emergence of one specific strain – HIV-1 subgroup M – in Léopoldville in the Belgian Congo (now Kinshasa in the Democratic Republic of the Congo) in the 1920s.

There are two types of HIV: HIV-1 and HIV-2. HIV-1 is more virulent, easily transmitted
---
Avian coronavirus is a species of virus from the genus Gammacoronavirus that infects birds; since 2018, all gammacoronaviruses which infect birds have been classified as this single species. The strain of avian coronavirus previously known as infectious bronchitis virus (IBV) is the only coronavirus that infects chickens. It causes avian infectious bronchitis, a highly infectious disease that affects the respiratory tract, gut, kidney and reproductive system. IBV affects the performance of 

Let’s finish with a final few questions.

In [30]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The war of currents is a term used to refer to a series of events in the '
 'history of the world that occurred in the late 19th century. The most famous '
 'of these events was the')


In [31]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The first man to walk on the moon was Neil Armstrong, who walked on the Moon '
 'in 1969 during the first crewed lunar mission.')


In [32]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The Space Shuttle was the most expensive project in the history of the US '
 'government. It cost about $10 billion to build.')


As we can see, the model can generate some decent answers.

#### Add a few more questions

### Observations:
### 1. Abstractive QA generates fluent natural language answers, unlike extractive QA which copies text directly
### 2. Answer quality depends heavily on retrieval - wrong context leads to confident but false answers (hallucination)
### 3. COVID-19 example is a clear failure case - no relevant context retrieved, generator invented a plausible-sounding but wrong answer
### 4. Well-known facts (moon landing, Space Shuttle cost) returned accurate answers when Wikipedia had relevant articles
### 5. Abstractive QA is more flexible than extractive but less transparent - you can't always trace where the answer came from
### 6. Dataset quality matters - random Wikipedia articles vs. history-filtered passages affects retrieval accuracy significantly